# Module 2 — FIBO Alignment and the Extension Ring

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS

---

## What this module teaches

Module 1 produced an ontology from competency questions. That ontology works — you can
query it, validate it, and defend every class in front of a reviewer. But it lives in
isolation. No other system in the financial services industry knows what `atlas:Customer`
means. If another team at your institution, or a regulator, or a vendor builds their own
ontology, there is no shared vocabulary connecting the two.

That shared vocabulary is FIBO — the Financial Industry Business Ontology. FIBO is
published by the Enterprise Data Management (EDM) Council, standardised by the Object
Management Group (OMG), and licensed under MIT. It defines the nouns and relationships
that the financial services industry has agreed on: parties, accounts, instruments,
legal entities, products, and the relationships among them.

This module teaches you to:

- Navigate FIBO's module structure (FND, BE, FBC, SEC, LOAN, DER) and know which
  module to open for which concept
- Bind your Module 1 classes to FIBO IRIs using `rdfs:subClassOf` — and understand
  why `subClassOf` is almost always correct and `owl:equivalentClass` is almost
  always wrong for institution-specific ontologies
- Identify the gaps — concepts FIBO deliberately does not cover — and select the
  appropriate extension standard for each (PROV-O, DCAT, SKOS, GLEIF, ISO 20022, BIAN)
- Produce two deliverables: `atlas-fibo-alignment.ttl` (the bindings) and
  `alignment-gaps.md` (the documented gaps with rationale)

## Why this module is longer than the others

FIBO is the part of the architecture that practicing teams are not doing daily. You are
likely encountering FIBO IRIs, the FIBO module structure, the difference between
Foundations and Business Entities, and the role of FIBO Securities for the first time.
This module deliberately allocates more time so you build genuine FIBO fluency rather
than copy-pasting IRIs you do not understand.

The goal is not to memorise FIBO. The goal is to know where to look, how to read what
you find, and how to make a binding decision you can defend.

## What this module does NOT do

- It does not import all of FIBO. FIBO is large (~1,500 classes across all modules).
  We import only the classes we bind to — approximately 8 FIBO IRIs.
- It does not use FIBO as a starting point. We already have our ontology from Module 1.
  FIBO is the alignment vocabulary, not the source vocabulary.
- It does not pretend FIBO is easy. FIBO has a real learning curve. This module
  respects that by walking through it module-by-module rather than hand-waving.

## Prerequisites

- Module 1 deliverable (`ontology/atlas-core.ttl` and `ontology/rationale.md`)
- Internet access from the SageMaker notebook to fetch FIBO module documentation
  (optional — the notebook works offline using the pinned FIBO IRIs)
- Amazon Bedrock enabled in us-east-1 (for the optional FIBO exploration exercise)

## Deliverables

- `ontology/atlas-fibo-alignment.ttl` — FIBO IRI bindings for every atlas-core class
  that has a FIBO counterpart
- `ontology/alignment-gaps.md` — every atlas-core class that does NOT have a FIBO
  counterpart, with rationale and chosen extension standard

## Architecture class for this module

**DETERMINISTIC.** Ontology alignment is a deterministic design decision: given the
same source ontology and the same FIBO version, a trained ontologist produces the same
bindings. The Bedrock LLM in this module is used only for exploration assistance —
it does not make alignment decisions.

## Cell 2 — Setup

Load the same shared utilities as Module 1. Confirm that `atlas-core.ttl` loads
correctly — this is the ontology we are aligning.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../notebooks/shared")

import rdflib
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD, SKOS
import pyshacl
import atlas_sparql
import atlas_validators

print(f"rdflib   : {rdflib.__version__}")
print(f"pyshacl  : {pyshacl.__version__}")

# Load the Module 1 ontology
CORE_PATH = Path("../ontology/atlas-core.ttl")
g_core = Graph()
g_core.parse(str(CORE_PATH), format="turtle")

core_classes = sorted(
    str(c).split("#")[-1]
    for c in g_core.subjects(RDF.type, OWL.Class)
)

print(f"\nModule 1 ontology loaded: {len(core_classes)} classes")
print(f"Source: {CORE_PATH}")
for i, c in enumerate(core_classes, 1):
    print(f"  {i:2d}. atlas:{c}")

## Cell 4 — What is FIBO and Why It Matters

### The problem FIBO solves

Every bank builds its own data model. The same concept — "customer", "account",
"transaction" — has a different name, a different structure, and different constraints
at every institution. When two systems need to talk to each other (a merger, a
regulatory report, a vendor integration), someone has to build a crosswalk. That
crosswalk is expensive, fragile, and usually wrong in edge cases.

FIBO is the industry's answer: a shared vocabulary that defines what these concepts
mean at a level of abstraction that every institution can agree on. It does not
replace your internal model. It provides a common reference point that your internal
model can be *aligned to*.

### What FIBO is, concretely

FIBO is a set of OWL ontology files published on GitHub at
`https://github.com/edmcouncil/fibo`. It is organised into modules:

| Module | Full Name | What Lives Here | When You Open It |
|--------|-----------|-----------------|------------------|
| **FND** | Foundations | Agents, parties, roles, organisations, time, place — the universal building blocks | Always. Every FIBO binding starts here. |
| **BE** | Business Entities | Legal entity structure, ownership, GLEIF/LEI integration | When you model institutional counterparties or legal structures |
| **FBC** | Financial Business and Commerce | Financial accounts, products, services, payment structures | When you model accounts, products, or transactions |
| **SEC** | Securities | Equity, debt, fund structures | When you model holdings or investment positions |
| **LOAN** | Loans | Loan structures, servicing, securitisation | When your use case involves loan-derived signals |
| **DER** | Derivatives | Options, swaps, structured products | Institutional or family-office use cases (not used in v1.0) |

### How to read a FIBO IRI

A FIBO IRI looks like this:

```
https://spec.edmcouncil.org/fibo/ontology/FND/Parties/Parties/IndependentParty
```

Reading it left to right:
- `spec.edmcouncil.org/fibo/ontology/` — the FIBO base (same for every class)
- `FND/` — the module (Foundations)
- `Parties/Parties/` — the sub-module path (Parties within Parties)
- `IndependentParty` — the class name

The prefix shorthand we use in this workshop:
```turtle
@prefix fibo-fnd-pty-pty: <https://spec.edmcouncil.org/fibo/ontology/FND/Parties/Parties/> .
```

So `fibo-fnd-pty-pty:IndependentParty` expands to the full IRI above.

### The FIBO version we pin to

FIBO publishes quarterly releases. This workshop pins to **FIBO 2024 Q3 Production
Release**. The "Production" qualifier matters — FIBO also publishes "Development"
releases with classes that may change. We bind only to Production-release IRIs.

### What FIBO is NOT

- It is not a database schema. You do not "implement FIBO" the way you implement a
  star schema. You *align to* FIBO.
- It is not complete. FIBO covers what the industry has agreed on. Your institution
  has concepts FIBO does not cover. That is expected and handled by the extension ring.
- It is not prescriptive about technology. FIBO does not care whether you use Neptune,
  Neo4j, or a triple store. It defines the vocabulary; you choose the infrastructure.

> **Skill to take away:** After this module, when someone says "we need to align to
> FIBO," you know what that means operationally: find the FIBO class that covers your
> concept, add an `rdfs:subClassOf` triple, and document the decision. That is the
> entire mechanical operation. The hard part is choosing the *right* FIBO class.

## Cell 5 — Walking Through FIBO Module by Module

We now walk through the FIBO modules in the order a practicing ontologist would
actually open them. For each module, we show:
1. What lives there
2. Which ATLAS class binds to it
3. The specific FIBO class we chose and why

---

### FND (Foundations) — Where Everything Starts

FIBO Foundations defines the universal building blocks: parties, roles, organisations,
dates, places, agreements. If your concept is "a person or organisation that does
something," it starts in FND.

**Key class: `fibo-fnd-pty-pty:IndependentParty`**

An IndependentParty is "a party that is not defined in terms of a role it plays."
This is FIBO's way of saying: a person or organisation that exists in its own right,
not just as a participant in a transaction.

Why we bind `atlas:Customer` here and not to `Person` or `LegalPerson`:
- A bank customer can be a natural person OR a small business (legal entity)
- If you bind to `Person`, you cannot model business accounts without a second class
- If you bind to `LegalPerson`, you lose natural persons
- `IndependentParty` covers both without forcing a premature choice

This is the most common FIBO binding mistake: choosing a class that is too specific.
The test: "Can every instance of my class also be an instance of this FIBO class?"
If the answer is "only some of them," you are binding too narrowly.

**Key class: `fibo-fnd-pty-rl:FunctionalRole`**

A FunctionalRole is "a role that is defined by the function performed by the party
playing it." We bind `atlas:Advisor` here because an advisor is a person playing
the wealth-advisory function — the same person could play other roles.

**Key class: `fibo-fnd-org-fm:OrganizationalSubUnit`**

We bind `atlas:LineOfBusiness` here. The bank's Consumer division, Wealth division,
and Commercial division are organisational sub-units in FIBO's sense.

---

### BE (Business Entities) — Legal Structure and Ownership

FIBO Business Entities defines legal entity structure, ownership chains, and the
GLEIF/LEI integration. You open this module when you need to model an institutional
counterparty with a Legal Entity Identifier.

**Key class: `fibo-be-le-lei:LegalPerson`**

We bind `atlas:LegalEntity` here. This class is introduced in Module 2 (not in
atlas-core.ttl) for the GLEIF demonstration. In v1.0, the wealth-signal use case
focuses on natural-person customers, but the architecture must support institutional
counterparties for the extension path.

**GLEIF integration pattern:**
```turtle
atlas:LegalEntity
    rdfs:subClassOf fibo-be-le-lei:LegalPerson ;
    rdfs:comment "An institutional party with a 20-character LEI." .
```

The LEI (Legal Entity Identifier) is a 20-character alphanumeric code governed by
GLEIF (Global Legal Entity Identifier Foundation). Every regulated financial entity
has one. FIBO BE provides the class structure; GLEIF provides the identifier.

---

### FBC (Financial Business and Commerce) — Accounts and Products

This is the largest single import in v1.0. FBC defines financial accounts, products,
services, and payment structures.

**Key class: `fibo-fbc-pas-fpas:FinancialAccount`**

We bind `atlas:Account` here. FIBO defines a FinancialAccount as "an account that
is maintained by a financial service provider." That is exactly what atlas:Account is.

**Key class: `fibo-fbc-pas-fpas:FinancialProduct`**

We bind `atlas:Product` here. A checking account, a brokerage account, a retirement
plan — these are all financial products in FIBO's sense.

---

### SEC (Securities) — Instruments and Positions

**Key class: `fibo-fbc-fi-ip:InvestmentPosition`**

We bind `atlas:Holding` here. A Holding is a customer's position in a financial
instrument — shares of stock, units of a fund, bonds. FIBO models this as an
InvestmentPosition.

Note: The IRI path is under FBC/FinancialInstruments rather than SEC. This is a
FIBO organisational choice — positions are in FBC because they relate to accounts,
while the instruments themselves are in SEC.

---

### LOAN and DER — Present but Not Imported in v1.0

**LOAN** covers loan structures, servicing, and securitisation. If your wealth-signal
use case includes loan-derived signals (e.g., business-loan payoff as a liquidity
signal), you would import LOAN classes here.

**DER** covers derivatives. Intentionally not imported in v1.0. The workshop notes
when a customer would import it: institutional or family-office wealth use cases
where structured products are relevant.

> **Proctor note:** Ask the room: "Which of these FIBO modules would you need to
> open for your institution's use case?" This surfaces whether participants are
> working on consumer banking (FND + FBC), institutional (FND + BE + SEC), or
> lending (FND + FBC + LOAN). The answer determines which extension-ring members
> they will need in their own alignment.

## Cell 6 — subClassOf vs equivalentClass: The Decision That Matters

When you bind your class to a FIBO class, you have two choices:

```turtle
# Option A: subClassOf
atlas:Customer rdfs:subClassOf fibo-fnd-pty-pty:IndependentParty .

# Option B: equivalentClass
atlas:Customer owl:equivalentClass fibo-fnd-pty-pty:IndependentParty .
```

**What each one means to a reasoner:**

| Binding | What it asserts | Consequence |
|---------|----------------|-------------|
| `rdfs:subClassOf` | Every atlas:Customer IS a fibo IndependentParty, but not every IndependentParty is an atlas:Customer | Safe. Your class is narrower than FIBO. A reasoner can infer that Customer instances are also IndependentParty instances. |
| `owl:equivalentClass` | atlas:Customer and fibo IndependentParty have IDENTICAL membership — every instance of one is an instance of the other | Dangerous. A reasoner will treat any FIBO IndependentParty as an atlas:Customer. If someone loads a FIBO dataset with IndependentParty instances that are not bank customers, your SHACL shapes will fire on them. |

**The rule for institution-specific ontologies:**

Almost always use `rdfs:subClassOf`. The reason: your institution adds constraints
that FIBO does not require. `atlas:Customer` requires "at least one active in-bank
relationship." FIBO's IndependentParty has no such requirement. The classes are not
equivalent — yours is a specialisation.

**When equivalentClass is correct:**

Only when your class adds zero constraints beyond what FIBO already defines. This is
rare in practice. The most common case: you created a class in Module 1 that turns
out to be *exactly* what FIBO already defines, with no institution-specific narrowing.
In that case, you should consider whether you need your own class at all — you could
just use the FIBO IRI directly.

**The most common mistake:**

Using `equivalentClass` because it "feels stronger" or "more aligned." It is not
stronger — it is a different assertion with different consequences. A reasoner that
sees `equivalentClass` will infer things you did not intend. Use `subClassOf` unless
you can prove the classes have identical membership.

> **Skill to take away:** When someone on your team proposes `owl:equivalentClass`,
> ask: "Does our class add any constraint that the FIBO class does not have?" If yes,
> it is `subClassOf`. If genuinely no, ask: "Then why do we have our own class?"

## Cell 7 — Three Worked Bindings in Turtle

The spec requires that we show three concrete bindings in detail rather than just
listing them. Here is what FIBO alignment actually looks like in Turtle syntax.

---

### Binding 1 — Customer as a FIBO IndependentParty

```turtle
atlas:Customer
    a owl:Class ;
    rdfs:subClassOf fibo-fnd-pty-pty:IndependentParty ;
    rdfs:label "Customer (in-bank)"@en ;
    rdfs:comment "A natural or legal person held by the institution
                  with at least one active in-bank relationship."@en .
```

**What this says:** Every atlas:Customer is also a FIBO IndependentParty. A SPARQL
query that asks for all IndependentParty instances will find our Customers. A SHACL
shape written against IndependentParty will also apply to our Customers (via RDFS
inference).

**What this does NOT say:** That every IndependentParty is a Customer. A vendor, a
regulator, a counterparty — these are all IndependentParty instances in FIBO, but
they are not atlas:Customer instances.

---

### Binding 2 — Account as a FIBO FinancialAccount

```turtle
atlas:Account
    a owl:Class ;
    rdfs:subClassOf fibo-fbc-pas-fpas:FinancialAccount ;
    rdfs:label "Account"@en .
```

**Why this is straightforward:** FIBO's FinancialAccount is defined as "an account
maintained by a financial service provider." Our Account is exactly that, narrowed
to accounts at *this* institution. The binding is clean.

---

### Binding 3 — Holding as a subclass of InvestmentPosition

```turtle
atlas:Holding
    a owl:Class ;
    rdfs:subClassOf fibo-fbc-fi-ip:InvestmentPosition ;
    rdfs:label "Holding"@en ;
    rdfs:comment "A customer's position in an instrument
                  observable from in-bank brokerage data."@en .
```

**Why "observable from in-bank brokerage data" matters:** This constraint narrows
the FIBO class. FIBO's InvestmentPosition covers any position anywhere. Ours covers
only positions we can see from inside the bank. That narrowing is why it is
`subClassOf` and not `equivalentClass`.

---

**Pattern to notice:** Every binding follows the same structure:
1. Declare the class (already done in atlas-core.ttl)
2. Add `rdfs:subClassOf <FIBO IRI>`
3. Add a comment explaining the narrowing

The comment is not optional. Six months from now, someone will ask "why did we bind
to IndependentParty and not Person?" The comment is where that answer lives.

In [ ]:
# Load the FIBO alignment file and verify bindings
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL

ATLAS = Namespace("https://github.com/your-org/atlas/ontology#")
FIBO_FND_PTY = Namespace("https://spec.edmcouncil.org/fibo/ontology/FND/Parties/Parties/")
FIBO_FND_RL = Namespace("https://spec.edmcouncil.org/fibo/ontology/FND/Parties/Roles/")
FIBO_FND_ORG = Namespace("https://spec.edmcouncil.org/fibo/ontology/FND/Organizations/FormalOrganizations/")
FIBO_FBC_FPAS = Namespace("https://spec.edmcouncil.org/fibo/ontology/FBC/ProductsAndServices/FinancialProductsAndServices/")
FIBO_FBC_IP = Namespace("https://spec.edmcouncil.org/fibo/ontology/FBC/FinancialInstruments/InstrumentPricing/")
FIBO_BE_LEI = Namespace("https://spec.edmcouncil.org/fibo/ontology/BE/LegalEntities/LegalPersons/")
PROV = Namespace("http://www.w3.org/ns/prov#")
DCAT = Namespace("http://www.w3.org/ns/dcat#")
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")

ALIGN_PATH = Path("../ontology/atlas-fibo-alignment.ttl")
g_align = Graph()
g_align.parse(str(ALIGN_PATH), format="turtle")

print(f"Alignment file loaded: {len(g_align)} triples")
print(f"Source: {ALIGN_PATH}")
print()

# Extract all subClassOf bindings
bindings = []
for s, p, o in g_align.triples((None, RDFS.subClassOf, None)):
    local = str(s).split("#")[-1]
    target = str(o)
    bindings.append((local, target))

print(f"FIBO/Extension bindings found: {len(bindings)}")
print()
print(f"{'ATLAS Class':<25} {'Bound To'}")
print("-" * 90)
for local, target in sorted(bindings):
    # Shorten the target for display
    short = target.replace("https://spec.edmcouncil.org/fibo/ontology/", "fibo:")
    short = short.replace("http://www.w3.org/ns/prov#", "prov:")
    short = short.replace("http://www.w3.org/ns/dcat#", "dcat:")
    short = short.replace("http://www.w3.org/2004/02/skos/core#", "skos:")
    print(f"  atlas:{local:<23} {short}")

## Cell 9 — The Extension Ring: What to Do When FIBO Is Silent

FIBO does not cover every concept a wealth-signal use case needs. That is by design —
FIBO covers what the industry has agreed on, not what any single institution needs.

The **extension ring** is the set of W3C and industry standards that real banks layer
around FIBO to cover the gaps. Each member of the ring has a specific job:

| Standard | What It Covers | ATLAS Classes That Use It |
|----------|---------------|---------------------------|
| **PROV-O** (W3C Provenance Ontology) | Who did what, when, from what source. Provenance of every promoted edge in the SLGD. | `AuditRecord` (subClassOf prov:Entity), `HumanReview` (subClassOf prov:Activity) |
| **DCAT v3** (W3C Data Catalog Vocabulary) | Catalog of federated data sources. What is connected, without reading R2RML mappings. | `DataSource` (subClassOf dcat:Dataset) |
| **SKOS** (W3C Simple Knowledge Organization System) | Code lists, taxonomies, controlled vocabularies. | `WealthSignalType` (subClassOf skos:Concept) |
| **GLEIF / LEI** | Legal entity identifiers for institutional counterparties. | `LegalEntity` (via FIBO BE, which integrates GLEIF) |
| **ISO 20022** | Transaction and payment message structure for stream-derived events. | Used at the edge in Module 4 Pattern C |
| **BIAN** | Service domain labels for crosswalking to bank operating-model terminology. | Used for labelling, not class binding |

### How to choose the right extension standard

When you encounter a concept FIBO does not cover, apply this decision tree:

1. **Is it about provenance?** (who produced this data, when, from what) → PROV-O
2. **Is it about data sources?** (what systems feed the graph) → DCAT
3. **Is it a controlled vocabulary?** (a closed list of types or categories) → SKOS
4. **Is it about legal entity identity?** → GLEIF via FIBO BE
5. **Is it about payment/transaction message structure?** → ISO 20022
6. **Is it about mapping to bank operating-model terminology?** → BIAN (labels only)
7. **None of the above?** → Bank-specific class, documented in alignment-gaps.md

### Why the extension ring exists as a concept

Without naming the ring explicitly, teams make one of two mistakes:
- They try to force everything into FIBO ("surely FIBO has a class for workflow
  routing?") and end up with incorrect bindings
- They give up on alignment entirely ("FIBO doesn't cover our stuff") and lose the
  interoperability benefit

The extension ring is the middle path: FIBO for what FIBO covers, named standards
for what they cover, and bank-specific classes (with documentation) for the rest.

> **Skill to take away:** When a colleague says "FIBO doesn't cover X, so we can't
> align," the answer is: "FIBO doesn't cover X, but PROV-O / DCAT / SKOS does.
> Which ring member fits?" The extension ring turns a dead end into a decision.

In [ ]:
# Classify every atlas-core class: FIBO-bound, extension-ring-bound, or bank-specific

# Expected bindings from atlas-fibo-alignment.ttl
FIBO_BOUND = {
    "Customer":       "fibo-fnd-pty-pty:IndependentParty",
    "Account":        "fibo-fbc-pas-fpas:FinancialAccount",
    "Holding":        "fibo-fbc-fi-ip:InvestmentPosition",
    "Transaction":    "fibo-fbc-pas-fpas:FinancialAccount (partial)",
    "Advisor":        "fibo-fnd-pty-rl:FunctionalRole",
    "LegalEntity":    "fibo-be-le-lei:LegalPerson",
    "Product":        "fibo-fbc-pas-fpas:FinancialProduct",
    "LineOfBusiness": "fibo-fnd-org-fm:OrganizationalSubUnit",
}

EXTENSION_RING_BOUND = {
    "DataSource":       ("DCAT v3",  "dcat:Dataset"),
    "AuditRecord":      ("PROV-O",   "prov:Entity"),
    "HumanReview":      ("PROV-O",   "prov:Activity"),
    "WealthSignalType": ("SKOS",     "skos:Concept"),
}

BANK_SPECIFIC = [
    "Household", "WealthSignal", "Eligibility", "Score",
    "RoutingDecision", "WorkflowStep", "HouseholdMembership",
    "ObservationWindow", "PreviousSurfacing",
]

print("ATLAS Class Alignment Classification")
print("=" * 70)
print()
print("FIBO-BOUND (direct subClassOf a FIBO Production class):")
for cls, fibo in sorted(FIBO_BOUND.items()):
    print(f"  atlas:{cls:<25} -> {fibo}")

print(f"\nEXTENSION-RING-BOUND (subClassOf a W3C/industry standard class):")
for cls, (std, target) in sorted(EXTENSION_RING_BOUND.items()):
    print(f"  atlas:{cls:<25} -> {target} ({std})")

print(f"\nBANK-SPECIFIC (no external binding; documented in alignment-gaps.md):")
for cls in sorted(BANK_SPECIFIC):
    print(f"  atlas:{cls}")

total = len(FIBO_BOUND) + len(EXTENSION_RING_BOUND) + len(BANK_SPECIFIC)
print(f"\nTotal classified: {total}")
print(f"  FIBO-bound:           {len(FIBO_BOUND)}")
print(f"  Extension-ring-bound: {len(EXTENSION_RING_BOUND)}")
print(f"  Bank-specific:        {len(BANK_SPECIFIC)}")

## Cell 11 — GLEIF/LEI Integration Demonstration

The Global Legal Entity Identifier Foundation (GLEIF) maintains the LEI registry —
a public database of 20-character identifiers for every regulated financial entity.
FIBO BE integrates GLEIF directly: the `fibo-be-le-lei:LegalPerson` class is designed
to carry an LEI.

This cell demonstrates a single LEI lookup and binding. In production, you would
batch-resolve LEIs for all institutional counterparties in your graph.

**Why this matters for the wealth-signal use case:**

When a business-sale liquidity signal fires (a large deposit from a business account),
the business entity behind that account may have an LEI. Binding it to FIBO BE via
GLEIF means the entity is interoperable with any other system that uses LEI —
regulatory reports, counterparty risk systems, AML (Anti-Money Laundering) checks.

The LEI lookup below uses a synthetic example. In production, you would call the
GLEIF API at `https://api.gleif.org/api/v1/lei-records/`.

In [ ]:
# GLEIF/LEI demonstration with a synthetic entity
# In production, this would call the GLEIF API.

from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD

ATLAS = Namespace("https://github.com/your-org/atlas/ontology#")
FIBO_BE = Namespace("https://spec.edmcouncil.org/fibo/ontology/BE/LegalEntities/LegalPersons/")
GLEIF = Namespace("https://www.gleif.org/ontology/Base/")

# Synthetic LEI for demonstration
SYNTHETIC_LEI = "5493001KJTIIGC8Y1R12"  # Not a real LEI
SYNTHETIC_ENTITY_NAME = "Acme Financial Holdings LLC"

g_lei = Graph()

# Create the entity
entity_uri = URIRef("https://github.com/your-org/atlas/instance#entity-acme")
g_lei.add((entity_uri, RDF.type, ATLAS.LegalEntity))
g_lei.add((entity_uri, RDFS.label, Literal(SYNTHETIC_ENTITY_NAME)))
g_lei.add((entity_uri, ATLAS.leiCode, Literal(SYNTHETIC_LEI, datatype=XSD.string)))

print("GLEIF/LEI Binding Demonstration")
print("=" * 60)
print()
print(f"Entity:  {SYNTHETIC_ENTITY_NAME}")
print(f"LEI:     {SYNTHETIC_LEI}")
print(f"Type:    atlas:LegalEntity")
print(f"Aligned: rdfs:subClassOf fibo-be-le-lei:LegalPerson")
print()
print("Turtle representation:")
print()
print(f'<{entity_uri}>')
print(f'    a atlas:LegalEntity ;')
print(f'    rdfs:label "{SYNTHETIC_ENTITY_NAME}" ;')
print(f'    atlas:leiCode "{SYNTHETIC_LEI}"^^xsd:string .')
print()
print("In production, the GLEIF API call would be:")
print(f"  GET https://api.gleif.org/api/v1/lei-records/{SYNTHETIC_LEI}")
print()
print("The response provides: legal name, jurisdiction, registration status,")
print("entity category, and the full ownership chain (Level 2 data).")

## Cell 13 — Documenting the Gaps

Nine of the eighteen atlas-core classes have no FIBO counterpart. This is not a
failure of alignment — it is the expected outcome when you model an operational
use case against an industry reference vocabulary.

FIBO covers what the industry agrees on. Your institution has concepts that are:
- **Operational** (workflow routing, human review steps) — FIBO models what things
  *are*, not what a bank *does with them*
- **ML-specific** (scores, SHAP explanations, model versions) — FIBO predates
  widespread ML in FSI decision-making
- **Institution-defined** (households, observation windows) — every bank defines
  these differently

For each gap, the `alignment-gaps.md` document records:
1. Why FIBO does not cover it (not a criticism — an explanation)
2. The chosen extension standard, if any
3. The rationale for the choice

This document is a deliverable. An architect who inherits this ontology six months
from now needs to understand why `atlas:WealthSignal` has no FIBO binding — and
whether that is a gap to close or a deliberate design decision.

> **The most common gotcha:** Confusing FIBO's IndependentParty with its more
> specific subclasses. A team that binds `atlas:Customer` to `fibo-fnd-pty-pty:Person`
> (instead of IndependentParty) will fail when a customer is also a small business.
> The fix is to bind to the more general class and document the choice.

In [ ]:
# Verify alignment-gaps.md exists and covers all unbound classes
from pathlib import Path

GAPS_PATH = Path("../ontology/alignment-gaps.md")

assert GAPS_PATH.exists(), f"alignment-gaps.md not found at {GAPS_PATH}"

gaps_content = GAPS_PATH.read_text()

# Check that every bank-specific class appears in the gaps document
missing_from_gaps = []
for cls in BANK_SPECIFIC:
    if f"atlas:{cls}" not in gaps_content and f"`atlas:{cls}`" not in gaps_content:
        missing_from_gaps.append(cls)

print("Alignment Gaps Document Check")
print("=" * 60)
print(f"File: {GAPS_PATH}")
print(f"Size: {len(gaps_content)} characters")
print()

if missing_from_gaps:
    print(f"[FAIL] Classes missing from alignment-gaps.md:")
    for cls in missing_from_gaps:
        print(f"  - atlas:{cls}")
else:
    print(f"[PASS] All {len(BANK_SPECIFIC)} bank-specific classes documented in alignment-gaps.md")
    for cls in sorted(BANK_SPECIFIC):
        print(f"  - atlas:{cls}")

## Cell 15 — Using Bedrock to Explore FIBO (Optional)

FIBO is large enough that finding the right class can take time. Bedrock can help
as an exploration assistant — not to make the binding decision, but to narrow the
search space.

**What the LLM does here:**
- Suggests candidate FIBO classes for a given concept
- Explains the difference between two candidate classes
- Identifies which FIBO module to look in

**What the LLM does NOT do:**
- Make the binding decision (you do that)
- Guarantee the IRI is correct (you verify against the FIBO GitHub)
- Know about your institution's specific constraints (you add those)

The cell below asks Bedrock to suggest FIBO candidates for a concept you provide.
Replace `YOUR_CONCEPT` with a class from your own domain.

In [ ]:
import json
import boto3
from botocore.exceptions import ClientError

_BEDROCK_MODEL = "us.anthropic.claude-sonnet-4-6"
bedrock = boto3.Session().client("bedrock-runtime", region_name="us-east-1")

# Replace with your own concept to explore
YOUR_CONCEPT = "Customer"  # <-- replace with your concept
YOUR_DESCRIPTION = "A natural or legal person who holds at least one active account with our bank"

fibo_prompt = f"""You are helping an FSI architect find the correct FIBO class to align to.

The architect has a class called '{YOUR_CONCEPT}' defined as:
  "{YOUR_DESCRIPTION}"

FIBO (Financial Industry Business Ontology) is organised into modules:
- FND (Foundations): parties, roles, organisations, time, place
- BE (Business Entities): legal entities, ownership, GLEIF/LEI
- FBC (Financial Business and Commerce): accounts, products, payments
- SEC (Securities): equity, debt, funds
- LOAN: loan structures
- DER: derivatives

Suggest the top 3 candidate FIBO classes this concept could align to.
For each candidate, provide:
1. The full FIBO IRI (from the 2024 Q3 Production Release)
2. Which FIBO module it is in
3. One sentence explaining why it might fit
4. One sentence explaining why it might NOT fit (the narrowing concern)

Do NOT make the final decision. Present the options for the architect to choose."""

try:
    response = bedrock.invoke_model(
        modelId=_BEDROCK_MODEL,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 800,
            "messages": [{"role": "user", "content": fibo_prompt}]
        })
    )
    result = json.loads(response["body"].read())
    fibo_suggestions = result["content"][0]["text"]
    
    print("FIBO Alignment Exploration — Bedrock Suggestions")
    print("=" * 60)
    print(f"Concept: {YOUR_CONCEPT}")
    print(f"Description: {YOUR_DESCRIPTION}")
    print()
    print(fibo_suggestions)
    print()
    print("-" * 60)
    print("IMPORTANT: Verify every IRI against the FIBO GitHub before using it.")
    print("The model may hallucinate IRIs that do not exist in the Production release.")
    print("Source of truth: https://github.com/edmcouncil/fibo")

except ClientError as exc:
    print(f"Bedrock call failed: {exc}")
    print("This cell is optional. The alignment file is already complete.")
    print("Continue to the next cell.")

## Cell 17 — Validate the Merged Ontology

The alignment file adds triples to the core ontology. We now load both files
together and verify that:
1. The merged graph parses without errors
2. Every core class either has a `rdfs:subClassOf` binding or is documented in gaps
3. No class has both a FIBO binding AND an entry in alignment-gaps.md (contradiction)
4. The SKOS codelists still load correctly alongside the alignment

In [ ]:
# Load all ontology files into a merged graph
from pathlib import Path
from rdflib import Graph
from rdflib.namespace import RDF, RDFS, OWL

ATLAS_NS = "https://github.com/your-org/atlas/ontology#"

g_merged = Graph()
g_merged.parse("../ontology/atlas-core.ttl", format="turtle")
g_merged.parse("../ontology/atlas-fibo-alignment.ttl", format="turtle")
g_merged.parse("../ontology/extensions/skos-codelists.ttl", format="turtle")

print(f"Merged graph: {len(g_merged)} triples")
print()

# Get all atlas: classes from the merged graph
all_classes = set()
for s in g_merged.subjects(RDF.type, OWL.Class):
    s_str = str(s)
    if ATLAS_NS in s_str:
        all_classes.add(s_str.split("#")[-1])

# Get all classes that have rdfs:subClassOf to a non-atlas target
bound_classes = set()
for s, p, o in g_merged.triples((None, RDFS.subClassOf, None)):
    s_str = str(s)
    o_str = str(o)
    if ATLAS_NS in s_str and ATLAS_NS not in o_str:
        bound_classes.add(s_str.split("#")[-1])

# Classes without external binding
unbound = all_classes - bound_classes

print(f"Total atlas: classes in merged graph: {len(all_classes)}")
print(f"Classes with external binding (FIBO or extension ring): {len(bound_classes)}")
print(f"Classes without external binding (bank-specific): {len(unbound)}")
print()

# Verify against our expected classification
expected_bound = set(list(FIBO_BOUND.keys()) + list(EXTENSION_RING_BOUND.keys()))
expected_unbound = set(BANK_SPECIFIC)

# Check for unexpected results
unexpected_bound = bound_classes - expected_bound
unexpected_unbound = unbound - expected_unbound

if unexpected_bound:
    print(f"[WARNING] Unexpectedly bound: {unexpected_bound}")
if unexpected_unbound:
    print(f"[WARNING] Unexpectedly unbound: {unexpected_unbound}")

if not unexpected_bound and not unexpected_unbound:
    print("[PASS] All classes classified as expected.")
    print()
    print("Bound classes:")
    for c in sorted(bound_classes):
        print(f"  atlas:{c}")
    print()
    print("Bank-specific (documented in alignment-gaps.md):")
    for c in sorted(unbound):
        print(f"  atlas:{c}")

## Cell 19 — Module 2 Validation Gate

This cell is the Module 2 validation gate. It must pass before you proceed to Module 3.

The gate checks:

1. **Alignment file parses** — `atlas-fibo-alignment.ttl` is valid Turtle
2. **Binding completeness** — every atlas-core class either has `rdfs:subClassOf`
   to an external IRI OR appears in `alignment-gaps.md`
3. **No contradictions** — no class has both a binding and a gaps entry
4. **Gaps document exists** — `alignment-gaps.md` is present and non-empty
5. **FIBO version documented** — the alignment file references the pinned FIBO version

In [ ]:
import sys
from pathlib import Path
from rdflib import Graph
from rdflib.namespace import RDF, RDFS, OWL

print("=" * 60)
print("MODULE 2 VALIDATION GATE")
print("=" * 60)

gate_pass = True
ATLAS_NS = "https://github.com/your-org/atlas/ontology#"

# --- Gate 1: Alignment file parses ---
try:
    g_test = Graph()
    g_test.parse("../ontology/atlas-fibo-alignment.ttl", format="turtle")
    print(f"[PASS] Gate 1 — atlas-fibo-alignment.ttl parses ({len(g_test)} triples)")
except Exception as exc:
    print(f"[FAIL] Gate 1 — Parse error: {exc}")
    gate_pass = False

# --- Gate 2: Binding completeness ---
# Load core to get the 18 original classes
g_core_check = Graph()
g_core_check.parse("../ontology/atlas-core.ttl", format="turtle")
core_class_names = set(
    str(c).split("#")[-1]
    for c in g_core_check.subjects(RDF.type, OWL.Class)
)

# Load alignment to get bound classes
g_align_check = Graph()
g_align_check.parse("../ontology/atlas-core.ttl", format="turtle")
g_align_check.parse("../ontology/atlas-fibo-alignment.ttl", format="turtle")

bound_check = set()
for s, p, o in g_align_check.triples((None, RDFS.subClassOf, None)):
    s_str = str(s)
    o_str = str(o)
    if ATLAS_NS in s_str and ATLAS_NS not in o_str:
        bound_check.add(s_str.split("#")[-1])

# Check gaps document
gaps_path = Path("../ontology/alignment-gaps.md")
gaps_content = gaps_path.read_text() if gaps_path.exists() else ""

unbound_check = core_class_names - bound_check
documented_in_gaps = set()
for cls in unbound_check:
    if f"atlas:{cls}" in gaps_content or f"`atlas:{cls}`" in gaps_content:
        documented_in_gaps.add(cls)

undocumented = unbound_check - documented_in_gaps

if not undocumented:
    print(f"[PASS] Gate 2 — All {len(core_class_names)} core classes accounted for")
    print(f"         Bound to external IRI: {len(bound_check)}")
    print(f"         Documented in gaps:    {len(documented_in_gaps)}")
else:
    print(f"[FAIL] Gate 2 — {len(undocumented)} class(es) neither bound nor documented:")
    for cls in sorted(undocumented):
        print(f"         atlas:{cls}")
    gate_pass = False

# --- Gate 3: No contradictions ---
contradictions = bound_check & documented_in_gaps
if not contradictions:
    print(f"[PASS] Gate 3 — No contradictions (no class is both bound and in gaps)")
else:
    print(f"[FAIL] Gate 3 — Contradictions found (both bound and in gaps):")
    for cls in sorted(contradictions):
        print(f"         atlas:{cls}")
    gate_pass = False

# --- Gate 4: Gaps document exists ---
if gaps_path.exists() and len(gaps_content) > 100:
    print(f"[PASS] Gate 4 — alignment-gaps.md present ({len(gaps_content)} chars)")
else:
    print(f"[FAIL] Gate 4 — alignment-gaps.md missing or empty")
    gate_pass = False

# --- Gate 5: FIBO version documented ---
align_content = Path("../ontology/atlas-fibo-alignment.ttl").read_text()
if "2024 Q3" in align_content:
    print(f"[PASS] Gate 5 — FIBO version (2024 Q3) documented in alignment file")
else:
    print(f"[FAIL] Gate 5 — FIBO version not found in alignment file")
    gate_pass = False

print()
if gate_pass:
    print("MODULE 2 VALIDATION: PASS")
    print("You may proceed to Module 3.")
else:
    print("MODULE 2 VALIDATION: FAIL")
    print("Fix the failing gate(s) above before proceeding to Module 3.")
    raise AssertionError("Module 2 validation gate failed. See output above.")

## Extending This to Your Data

### Which FIBO modules to import for which lines of business

| Your Line of Business | FIBO Modules to Import | Why |
|---|---|---|
| Consumer banking (deposits, cards, mortgages) | FND + FBC | Parties, accounts, products, payments |
| Wealth management | FND + FBC + SEC | Add securities positions and fund structures |
| Commercial/institutional banking | FND + BE + FBC | Add legal entity structure and ownership |
| Lending | FND + FBC + LOAN | Add loan structures and servicing |
| Capital markets | FND + BE + FBC + SEC + DER | Full stack for institutional trading |

### How to handle the case where two FIBO classes both look like candidates

This happens frequently. The workshop's recommendation:

1. **Bind to the more specific class** (the one that is a subclass of the other)
2. **Document the choice** in a comment on the binding triple
3. **Document the alternative** you considered and why you rejected it

Example: For `atlas:Customer`, both `IndependentParty` and `Person` are candidates.
`Person` is a subclass of `IndependentParty`. We chose `IndependentParty` (the more
general class) because our Customer can be either a natural person or a small business.
If we had chosen `Person`, business-account customers would violate the binding.

### The most common gotcha

**Confusing FIBO's IndependentParty with its more specific subclasses.**

FIBO's party hierarchy:
```
IndependentParty
  ├── Person (natural person)
  └── Organization
        ├── FormalOrganization
        └── InformalOrganization
```

If you bind to `Person` and later discover that some of your customers are small
businesses (which are Organizations, not Persons), your binding is wrong and your
SHACL shapes will fire on every business-account customer.

The fix: bind to `IndependentParty` from the start. You can always narrow later
(by adding a more specific subclass for natural-person customers). You cannot
easily widen a binding that is already in production.

### Checklist for FIBO alignment in your own context

- [ ] For each class in your ontology, identify the FIBO module that covers it
- [ ] Find the most specific FIBO class that ALL your instances satisfy
- [ ] Add `rdfs:subClassOf` (not `owl:equivalentClass` unless you can prove identical membership)
- [ ] Add a comment explaining the narrowing (what constraint you add that FIBO does not)
- [ ] For classes with no FIBO counterpart, document in alignment-gaps.md
- [ ] For each gap, identify the extension-ring member (PROV-O, DCAT, SKOS) or mark bank-specific
- [ ] Verify the FIBO IRI exists in the Production release (not just Development)

## What Changed

Module 2 added the following to the ATLAS architecture:

| Artifact | Location | Description |
|----------|----------|-------------|
| `atlas-fibo-alignment.ttl` | `ontology/` | FIBO IRI bindings (rdfs:subClassOf) for 8 core classes + 4 extension-ring bindings + 3 new classes (LegalEntity, Product, LineOfBusiness) |
| `alignment-gaps.md` | `ontology/` | Documents 9 bank-specific classes with rationale for why FIBO does not cover them |
| GLEIF/LEI demonstration | This notebook | Shows how to bind an institutional counterparty to FIBO BE via LEI |

**New classes introduced in Module 2** (not in atlas-core.ttl but declared in the alignment file):
- `atlas:LegalEntity` — for institutional counterparties with LEI
- `atlas:Product` — for financial products offered by the institution
- `atlas:LineOfBusiness` — for the bank's internal organisational units

**What Module 3 builds on this:**

Module 3 takes the aligned ontology (atlas-core.ttl + atlas-fibo-alignment.ttl) and
loads it into a running Amazon Neptune cluster. Specifically, it deploys two Neptune
clusters (LGD and SLGD), loads the ontology into the SLGD, and runs SPARQL discovery
queries that confirm the FIBO-aligned classes are queryable.

The key question Module 3 answers: does the ontology work as a physical graph, not
just as a Turtle file on disk?